# Poker Hand Prediction

Đây là pipeline Machine Learning cơ bản sử dụng Cross-Validation và GridSearch.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Tải dữ liệu

In [2]:
# Đường dẫn tương đối từ thư mục models tới data
train_path = r'..\data\train.csv'
test_path = r'..\data\test.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

X_train = df_train.drop('Class', axis=1)
y_train = df_train['Class']

X_test = df_test.drop('Class', axis=1)
y_test = df_test['Class']

Train shape: (25010, 11)
Test shape: (1000000, 11)


## 2. Xây dựng Pipeline & Cross-Validation

In [3]:
# Pipeline gồm 2 bước: Scale dữ liệu và Model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

# Thiết lập GridSearch để tìm hyperparameter tốt nhất với Cross Validation 3 folds
param_grid = {
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [10, 20, None]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

## 3. Huấn luyện mô hình

In [4]:
# Do tập train có 25k dòng, việc train sẽ tương đối nhanh
print("Đang huấn luyện mô hình...")
grid_search.fit(X_train, y_train)
print("Hoàn thành!")
print("Tham số tốt nhất:", grid_search.best_params_)
print("Điểm CV tốt nhất:", grid_search.best_score_)

Đang huấn luyện mô hình...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Hoàn thành!
Tham số tốt nhất: {'clf__max_depth': None, 'clf__n_estimators': 100}
Điểm CV tốt nhất: 0.5996401890270009


## 4. Đánh giá trên tập Test

In [5]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy trên tập test:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy trên tập test: 0.608874

Classification Report:
               precision    recall  f1-score   support

           0       0.63      0.80      0.71    501209
           1       0.57      0.49      0.52    422498
           2       0.39      0.00      0.01     47622
           3       0.49      0.00      0.01     21121
           4       0.22      0.00      0.00      3885
           5       1.00      0.01      0.01      1996
           6       0.00      0.00      0.00      1424
           7       0.00      0.00      0.00       230
           8       0.00      0.00      0.00        12
           9       0.00      0.00      0.00         3

    accuracy                           0.61   1000000
   macro avg       0.33      0.13      0.13   1000000
weighted avg       0.59      0.61      0.58   1000000

